![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [ ]:
# QuantBook Analysis Tool 
# For more information see [https://www.quantconnect.com/docs/v2/our-platform/research/getting-started]
import matplotlib.pyplot as plt
import mplfinance       # for candlestick
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

RANGE_THRESHOLD = 51.698
start_date = datetime(2022, 4, 1)
end_date = datetime(2025, 12, 31)


## Data

### Questions
 - Do implied vol measures add incremental information over lagged realized volatility?

In [3]:

qb = QuantBook()

spx_dly = qb.add_index("SPX", Resolution.DAILY)

spx_dly_hist = qb.history(spx_dly.symbol,  start_date, end_date)
spx_dly_hist['date'] = spx_dly_hist.index.get_level_values('time').date
spx_dly_hist['range'] = spx_dly_hist['high'] - spx_dly_hist['low']
spx_dly_hist['5d_avg_range'] = spx_dly_hist['range'].shift(1).rolling(5).mean()
spx_dly_hist['prior_range'] = spx_dly_hist['range'].shift(1)
spx_dly_hist['range_compression'] = (
    spx_dly_hist['prior_range'] / spx_dly_hist['5d_avg_range']
)
spx_dly_hist['regime_target'] = spx_dly_hist['range'] >= RANGE_THRESHOLD
spx_dly_hist['ret'] = np.log(spx_dly_hist['close'] / spx_dly_hist['close'].shift(1))  # NOTE: log return
spx_dly_hist['prior_abs_ret'] = spx_dly_hist['ret'].shift(1).abs()
spx_dly_hist['rv_5d'] = spx_dly_hist['ret'].shift(1).rolling(5).std()
spx_dly_hist['rv_5d_ann'] = spx_dly_hist['rv_5d'] * np.sqrt(252) * 100   
spx_dly_hist['gap'] = spx_dly_hist['open'] - spx_dly_hist['close'].shift(1)
spx_dly_hist['gap_mag'] = (
    (spx_dly_hist['open'] - spx_dly_hist['close'].shift(1)) / spx_dly_hist['close'].shift(1)
).abs()
spx_dly_hist = spx_dly_hist.dropna()
spx_dly_hist = spx_dly_hist.reset_index(level=0, drop=True)
spx_dly_hist.index.name = 'time'

spx_dly_hist.head(20)


In [4]:

qb = QuantBook()
vix_dly = qb.add_index("VIX", Resolution.DAILY)
vix9d_dly = qb.add_index("VIX9D", Resolution.DAILY)

vix_dly_hist = qb.history(vix_dly.symbol,  start_date, end_date)
vix_dly_hist['date'] = vix_dly_hist.index.get_level_values('time').date
vix_dly_hist['prior_vix_close'] = vix_dly_hist['close'].shift(1)
vix_dly_hist['vix_delta'] = vix_dly_hist['close'].shift(1) - vix_dly_hist['close'].shift(2)

vix9d_dly_hist = qb.history(vix9d_dly.symbol,  start_date, end_date)
vix9d_dly_hist['date'] = vix9d_dly_hist.index.get_level_values('time').date
vix9d_dly_hist['prior_vix9d_close'] = vix9d_dly_hist['close'].shift(1)

vix_ft = (
    vix_dly_hist.reset_index().merge(vix9d_dly_hist[['date', 'prior_vix9d_close']], on='date', how='left')
).set_index(['time'])

vix_ft['prior_slope'] = vix_ft['prior_vix9d_close'] - vix_ft['prior_vix_close']
vix_ft = vix_ft[['date', 'prior_vix_close', 'prior_vix9d_close', 'prior_slope', 'vix_delta']]

# vix_ft = (
#     spx_dly_hist[['date', 'range', 'regime_target']].merge(vix_ft[['date', 'prior_vix_close', 'prior_vix9d_close', 'prior_slope']], on='date', how='left')
# )
# vix_ft = vix_ft.set_index('date', drop=True)
# vix_ft.index = pd.to_datetime(vix_ft.index)
# vix_ft.index.name = 'date'



In [5]:
combo_ft = (
    spx_dly_hist[['date', 'range', 'regime_target', '5d_avg_range', 'prior_range', 'rv_5d_ann', 'gap_mag', 'prior_abs_ret', 'range_compression']]
        .merge(vix_ft[['date', 'prior_vix_close', 'prior_vix9d_close', 'prior_slope', 'vix_delta']], on='date', how='left')
)
combo_ft = combo_ft.set_index('date', drop=True)
combo_ft.index = pd.to_datetime(combo_ft.index)
vix_ft.index.name = 'date'

combo_ft['iv_rv_spread'] = combo_ft['prior_vix9d_close'] - combo_ft['rv_5d_ann']
combo_ft['short_vol_ratio'] = (
    combo_ft['prior_vix9d_close'] / combo_ft['prior_vix_close']
)

combo_ft.head(20)


In [6]:
def run_experiment(train, test, fts):
    X_train = train[fts]
    y_train = train['regime_target']

    X_test = test[fts]
    y_test = test['regime_target']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(penalty='l2', C=1.0)
    model.fit(X_train_scaled, y_train)

    train_probs = model.predict_proba(X_train_scaled)[:, 1]
    test_probs = model.predict_proba(X_test_scaled)[:, 1]

    results = {
        'train_log_loss': log_loss(y_train, train_probs),
        'train_brier': brier_score_loss(y_train, train_probs),
        'train_AUC': roc_auc_score(y_train, train_probs),
        'test_log_loss': log_loss(y_test, test_probs),
        'test_brier': brier_score_loss(y_test, test_probs),
        'test_AUC': roc_auc_score(y_test, test_probs),
        'train_probs': train_probs,
        'test_probs': test_probs,
        'y_train': y_train,
        'y_test': y_test
    }

    return results


## Realized Volatility: Prior Day Range & 5d Avg Range

In [7]:
train = spx_dly_hist.loc['2022-04-01':'2023-12-31']
test  = spx_dly_hist.loc['2024-01-01':'2025-12-31']

In [8]:
fts = ['prior_range', '5d_avg_range']
real_vol_results = run_experiment(train, test, fts)

In [9]:
print(f"Realized Vol Train Log Loss: {real_vol_results['train_log_loss']}")
print(f"Realized Vol Train Brier Score: {real_vol_results['train_brier']}")
print(f"Realized Vol Train AUC: {real_vol_results['train_AUC']}")

In [10]:
train_df = pd.DataFrame({
    'prob': real_vol_results['train_probs'],
    'actual': real_vol_results['y_train']
})

train_df['quintile'] = pd.qcut(train_df['prob'], 5, labels=False)

lift_table = train_df.groupby('quintile').agg(
    mean_prob=('prob', 'mean'),
    realized_freq=('actual', 'mean'),
    count=('actual', 'count')
)

print(lift_table)

In [11]:
prob_true, prob_pred = calibration_curve(real_vol_results['y_train'], real_vol_results['train_probs'], n_bins=10)

plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")
plt.ylabel("Realized Frequency")
plt.title("Training Calibration")
plt.show()

### Test Data

In [12]:
print(f"Realized Vol Train Log Loss: {real_vol_results['test_log_loss']}")
print(f"Realized Vol Train Brier Score: {real_vol_results['test_brier']}")
print(f"Realized Vol Train AUC: {real_vol_results['test_AUC']}")

In [13]:
test_df = pd.DataFrame({
    'prob': real_vol_results['test_probs'],
    'actual': real_vol_results['y_test']
})

test_df['quintile'] = pd.qcut(test_df['prob'], 5, labels=False)

test_lift_table = test_df.groupby('quintile').agg(
    mean_prob=('prob', 'mean'),
    realized_freq=('actual', 'mean'),
    count=('actual', 'count')
)

print(test_lift_table)

In [14]:
prob_true, prob_pred = calibration_curve(real_vol_results['y_test'], real_vol_results['test_probs'], n_bins=10)

plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")
plt.ylabel("Realized Frequency")
plt.title("Testing Calibration")
plt.show()

## Implied Volatility

In [15]:
vix_ft_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope']
vix_train = combo_ft.loc['2022-04-01':'2023-12-31']
vix_test  = combo_ft.loc['2024-01-01':'2025-12-31']

imp_vol_results = run_experiment(vix_train, vix_test, vix_ft_cols)

In [16]:
print(f"Implied Vol Train Log Loss: {imp_vol_results['train_log_loss']}")
print(f"Implied Vol Train Brier Score: {imp_vol_results['train_brier']}")
print(f"Implied Vol Train AUC: {imp_vol_results['train_AUC']}")

In [17]:
train_df = pd.DataFrame({
    'prob': imp_vol_results['train_probs'],
    'actual': imp_vol_results['y_train']
})

train_df['quintile'] = pd.qcut(train_df['prob'], 5, labels=False)

lift_table = train_df.groupby('quintile').agg(
    mean_prob=('prob', 'mean'),
    realized_freq=('actual', 'mean'),
    count=('actual', 'count')
)

print(lift_table)

In [18]:
prob_true, prob_pred = calibration_curve(imp_vol_results['y_train'], imp_vol_results['train_probs'], n_bins=10)

plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")
plt.ylabel("Realized Frequency")
plt.title("Training Calibration")
plt.show()

### Test Data

In [19]:
p_base = y_test.mean()
print("Test base rate:", p_base)

In [ ]:
print(f"Implied Vol Train Log Loss: {imp_vol_results['test_log_loss']}")
print(f"Implied Vol Train Brier Score: {imp_vol_results['test_brier']}")
print(f"Implied Vol Train AUC: {imp_vol_results['test_AUC']}")

In [ ]:
test_df = pd.DataFrame({
    'prob': imp_vol_results['test_probs'],
    'actual': imp_vol_results['y_test']
})

test_df['quintile'] = pd.qcut(test_df['prob'], 5, labels=False)

test_lift_table = test_df.groupby('quintile').agg(
    mean_prob=('prob', 'mean'),
    realized_freq=('actual', 'mean'),
    count=('actual', 'count')
)

print(test_lift_table)

In [ ]:
prob_true, prob_pred = calibration_curve(imp_vol_results['y_test'], imp_vol_results['test_probs'], n_bins=10)

plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")
plt.ylabel("Realized Frequency")
plt.title("Testing Calibration")
plt.show()

## Combined Features

In [ ]:
combo_ft_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope', '5d_avg_range', 'prior_range']
combo_train = combo_ft.loc['2022-04-01':'2023-12-31']
combo_test  = combo_ft.loc['2024-01-01':'2025-12-31']

combo_results = run_experiment(combo_train, combo_test, combo_ft_cols)

In [ ]:
print(f"Combo Train Log Loss: {combo_results['train_log_loss']}")
print(f"Combo Train Brier Score: {combo_results['train_brier']}")
print(f"Combo Train AUC: {combo_results['train_AUC']}")

In [ ]:
train_df = pd.DataFrame({
    'prob': combo_results['train_probs'],
    'actual': combo_results['y_train']
})

train_df['quintile'] = pd.qcut(train_df['prob'], 5, labels=False)

lift_table = train_df.groupby('quintile').agg(
    mean_prob=('prob', 'mean'),
    realized_freq=('actual', 'mean'),
    count=('actual', 'count')
)

print(lift_table)

In [ ]:
prob_true, prob_pred = calibration_curve(combo_results['y_train'], combo_results['train_probs'], n_bins=10)

plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")
plt.ylabel("Realized Frequency")
plt.title("Training Calibration")
plt.show()

### Test Data

In [ ]:
p_base = combo_y_test.mean()
print("Test base rate:", p_base)

In [ ]:
print(f"Combo Test Log Loss: {combo_results['test_log_loss']}")
print(f"Combo Test Brier Score: {combo_results['test_brier']}")
print(f"Combo Test AUC: {combo_results['test_AUC']}")

In [42]:
test_df = pd.DataFrame({
    'prob': combo_results['test_probs'],
    'actual': combo_results['y_test']
})

test_df['quintile'] = pd.qcut(test_df['prob'], 5, labels=False)

test_lift_table = test_df.groupby('quintile').agg(
    mean_prob=('prob', 'mean'),
    realized_freq=('actual', 'mean'),
    count=('actual', 'count')
)

print(test_lift_table)

In [43]:
prob_true, prob_pred = calibration_curve(combo_results['y_test'], combo_results['test_probs'], n_bins=10)

plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")
plt.ylabel("Realized Frequency")
plt.title("Testing Calibration")
plt.show()

## Additional Features

In [ ]:
qb = QuantBook()

spx_hrly = qb.add_index("SPX", Resolution.HOUR)
vix_hrly = qb.add_index("VIX", Resolution.HOUR)
vix9d_hrly = qb.add_index("VIX9D", Resolution.HOUR)

spx_hrly_hist = qb.history(spx_hrly.symbol,  start_date, end_date)
vix_hrly_hist = qb.history(vix_hrly.symbol,  start_date, end_date)
vix9d_hrly_hist = qb.history(vix9d_hrly.symbol,  start_date, end_date)
spx_hrly_hist.head(10)

In [46]:
time_idx = spx_hrly_hist.index.get_level_values("time")
time_idx

In [47]:
mask = (time_idx.hour >= 9) & (time_idx.hour <= 15) # NOTE: excludes the final hour of trading
spx_intraday = spx_hrly_hist[mask]
spx_intraday = spx_intraday.reset_index(level="symbol", drop=True)
spx_intraday['date'] = spx_intraday.index.date
spx_intraday.head(20)

In [48]:
spx_intraday_range = spx_intraday.groupby('date').agg(high_max=('high', 'max'), low_min=('low', 'min'))
spx_intraday_range['range'] = spx_intraday_range['high_max'] - spx_intraday_range['low_min']
spx_intraday_range.head(20)

In [49]:
addl_ft = combo_ft.join(
    spx_intraday_range[['range']].add_suffix('_intraday'),
    how='left'
)
addl_ft.head()

### IV-RV Spread

In [50]:
iv_rv_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope', '5d_avg_range', 'prior_range', 'iv_rv_spread']
iv_rv_train = addl_ft.loc['2022-04-01':'2023-12-31']
iv_rv_test  = addl_ft.loc['2024-01-01':'2025-12-31']

iv_rv_results = run_experiment(iv_rv_train, iv_rv_test, iv_rv_cols)

In [ ]:
print(f"IV-RV Spread Test Log Loss: {iv_rv_results['test_log_loss']}")
print(f"IV-RV Spread Test Brier Score: {iv_rv_results['test_brier']}")
print(f"IV-RV Spread Test AUC: {iv_rv_results['test_AUC']}")

### Overnight Gap

In [ ]:
gap_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope', '5d_avg_range', 'prior_range', 'gap_mag']
gap_train = addl_ft.loc['2022-04-01':'2023-12-31']
gap_test  = addl_ft.loc['2024-01-01':'2025-12-31']

gap_results = run_experiment(gap_train, gap_test, gap_cols)

In [55]:
print(f"Gap Spread Test Log Loss: {gap_results['test_log_loss']}")
print(f"Gap Spread Test Brier Score: {gap_results['test_brier']}")
print(f"Gap Spread Test AUC: {gap_results['test_AUC']}")

### prior_abs_ret

In [57]:
prior_abs_ret_train = combo_ft.loc['2022-04-01':'2023-12-31']
prior_abs_ret_test  = combo_ft.loc['2024-01-01':'2025-12-31']
prior_abs_ret_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope', '5d_avg_range', 'prior_range', 'prior_abs_ret']

prior_abs_ret_results = run_experiment(prior_abs_ret_train, prior_abs_ret_test, prior_abs_ret_cols)

In [58]:
print(f"prior_abs_ret Spread Test Log Loss: {prior_abs_ret_results['test_log_loss']}")
print(f"prior_abs_ret Spread Test Brier Score: {prior_abs_ret_results['test_brier']}")
print(f"prior_abs_ret Spread Test AUC: {prior_abs_ret_results['test_AUC']}")

### Range Compression

In [ ]:
range_compression_train = combo_ft.loc['2022-04-01':'2023-12-31']
range_compression_test  = combo_ft.loc['2024-01-01':'2025-12-31']
range_compression_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope', '5d_avg_range', 'prior_range', 'range_compression']

range_compression_results = run_experiment(range_compression_train, range_compression_test, range_compression_cols)

In [ ]:
print(f"range_compression Spread Test Log Loss: {range_compression_results['test_log_loss']}")
print(f"range_compression Spread Test Brier Score: {range_compression_results['test_brier']}")
print(f"range_compression Spread Test AUC: {range_compression_results['test_AUC']}")

### VIX Delta

In [61]:
vix_delta_train = combo_ft.loc['2022-04-01':'2023-12-31']
vix_delta_test  = combo_ft.loc['2024-01-01':'2025-12-31']
vix_delta_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope', '5d_avg_range', 'prior_range', 'vix_delta']

vix_delta_results = run_experiment(vix_delta_train, vix_delta_test, vix_delta_cols)

In [62]:
print(f"vix_delta Spread Test Log Loss: {vix_delta_results['test_log_loss']}")
print(f"vix_delta Spread Test Brier Score: {vix_delta_results['test_brier']}")
print(f"vix_delta Spread Test AUC: {vix_delta_results['test_AUC']}")

### Short Vol Ratio

In [ ]:
short_vol_ratio_train = combo_ft.loc['2022-04-01':'2023-12-31']
short_vol_ratio_test  = combo_ft.loc['2024-01-01':'2025-12-31']
short_vol_ratio_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope', '5d_avg_range', 'prior_range', 'short_vol_ratio']

short_vol_ratio_results = run_experiment(short_vol_ratio_train, short_vol_ratio_test, short_vol_ratio_cols)

In [64]:
print(f"short_vol_ratio Spread Test Log Loss: {short_vol_ratio_results['test_log_loss']}")
print(f"short_vol_ratio Spread Test Brier Score: {short_vol_ratio_results['test_brier']}")
print(f"short_vol_ratio Spread Test AUC: {short_vol_ratio_results['test_AUC']}")

### All Features Set

In [ ]:
all_ft_train = combo_ft.loc['2022-04-01':'2023-12-31']
all_ft_test  = combo_ft.loc['2024-01-01':'2025-12-31']
all_ft_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope', '5d_avg_range', 'prior_range', 'prior_abs_ret', 'gap_mag']

all_ft_results = run_experiment(all_ft_train, all_ft_test, all_ft_cols)

In [66]:
print(f"all_ft Spread Test Log Loss: {all_ft_results['test_log_loss']}")
print(f"all_ft Spread Test Brier Score: {all_ft_results['test_brier']}")
print(f"all_ft Spread Test AUC: {all_ft_results['test_AUC']}")

In [ ]:
all_ft_test_df = pd.DataFrame({
    'prob': all_ft_results['test_probs'],
    'actual': all_ft_results['y_test']
})

all_ft_test_df['quintile'] = pd.qcut(all_ft_test_df['prob'], 5, labels=False)

all_ft_test_lift_table = all_ft_test_df.groupby('quintile').agg(
    mean_prob=('prob', 'mean'),
    realized_freq=('actual', 'mean'),
    count=('actual', 'count')
)

print(all_ft_test_lift_table)


In [69]:
all_ft_prob_true, all_ft_prob_pred = calibration_curve(all_ft_results['y_test'], all_ft_results['test_probs'], n_bins=10)

plt.plot(all_ft_prob_pred, all_ft_prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")
plt.ylabel("Realized Frequency")
plt.title("all_ft Test Calibration")
plt.show()


### Ablation Tests

In [ ]:
all_ft_train = combo_ft.loc['2022-04-01':'2023-12-31']
all_ft_test  = combo_ft.loc['2024-01-01':'2025-12-31']
all_ft_cols = ['prior_vix_close', 'prior_vix9d_close', 'prior_slope', '5d_avg_range', 'prior_range', 'prior_abs_ret', 'gap_mag']

all_ft_results = run_experiment(all_ft_train, all_ft_test, all_ft_cols)

In [80]:
for ft in all_ft_cols:
    abl_ft = [f for f in all_ft_cols if f != ft]
    cols = abl_ft + ['regime_target']

    exp_result = run_experiment(all_ft_train[cols], all_ft_test[cols], abl_ft)

    print('')
    print(f'Removing: {ft}')
    print(f"all_ft Spread Test Log Loss: {exp_result['test_log_loss']}")
    print(f"all_ft Spread Test Brier Score: {exp_result['test_brier']}")
    print(f"all_ft Spread Test AUC: {exp_result['test_AUC']}")
    

## Remove Redundant Features

In [81]:
final_ft_train = combo_ft.loc['2022-04-01':'2023-12-31']
final_ft_test  = combo_ft.loc['2024-01-01':'2025-12-31']
final_ft_cols = ['prior_slope', '5d_avg_range', 'prior_abs_ret', 'gap_mag']

final_ft_results = run_experiment(final_ft_train, final_ft_test, final_ft_cols)

In [82]:
print(f"final_ft Spread Test Log Loss: {final_ft_results['test_log_loss']}")
print(f"final_ft Spread Test Brier Score: {final_ft_results['test_brier']}")
print(f"final_ft Spread Test AUC: {final_ft_results['test_AUC']}")

In [83]:
final_ft_test_df = pd.DataFrame({
    'prob': final_ft_results['test_probs'],
    'actual': final_ft_results['y_test']
})

final_ft_test_df['quintile'] = pd.qcut(final_ft_test_df['prob'], 5, labels=False)

final_ft_lift_table = final_ft_test_df.groupby('quintile').agg(
    mean_prob=('prob', 'mean'),
    realized_freq=('actual', 'mean'),
    count=('actual', 'count')
)

print(final_ft_lift_table)

In [ ]:
final_ft_prob_true, final_ft_prob_pred = calibration_curve(final_ft_results['y_test'], final_ft_results['test_probs'], n_bins=10)

plt.plot(final_ft_prob_pred, final_ft_prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")a
plt.ylabel("Realized Frequency")
plt.title("final_ft Test Calibration")
plt.show()


## Post Charts

In [ ]:
prob_true, prob_pred = calibration_curve(real_vol_results['y_test'], real_vol_results['test_probs'], n_bins=10)
prob_true, prob_pred = calibration_curve(imp_vol_results['y_test'], imp_vol_results['test_probs'], n_bins=10)
prob_true, prob_pred = calibration_curve(combo_results['y_test'], combo_results['test_probs'], n_bins=10)
all_ft_prob_true, all_ft_prob_pred = calibration_curve(all_ft_results['y_test'], all_ft_results['test_probs'], n_bins=10)
final_ft_prob_true, final_ft_prob_pred = calibration_curve(final_ft_results['y_test'], final_ft_results['test_probs'], n_bins=10)

plt.plot(final_ft_prob_pred, final_ft_prob_true, marker='o')
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")a
plt.ylabel("Realized Frequency")
plt.title("final_ft Test Calibration")
plt.show()


In [85]:
rv_true, rv_pred = calibration_curve(real_vol_results['y_test'],
                                     real_vol_results['test_probs'], n_bins=10)

iv_true, iv_pred = calibration_curve(imp_vol_results['y_test'],
                                     imp_vol_results['test_probs'], n_bins=10)

combo_true, combo_pred = calibration_curve(combo_results['y_test'],
                                           combo_results['test_probs'], n_bins=10)

all_ft_true, all_ft_pred = calibration_curve(all_ft_results['y_test'],
                                             all_ft_results['test_probs'], n_bins=10)

final_ft_true, final_ft_pred = calibration_curve(final_ft_results['y_test'],
                                                 final_ft_results['test_probs'], n_bins=10)

# Plot
plt.figure(figsize=(8,6))

plt.plot(rv_pred, rv_true, marker='o', label='Real Vol Model')
plt.plot(iv_pred, iv_true, marker='o', label='Implied Vol Model')
plt.plot(combo_pred, combo_true, marker='o', label='Combined Model')
plt.plot(all_ft_pred, all_ft_true, marker='o', label='All Features Model')
plt.plot(final_ft_pred, final_ft_true, marker='o', label='Final Features Model')

# Perfect calibration reference
plt.plot([0,1], [0,1], '--', label='Perfect Calibration')

plt.xlabel("Predicted Probability")
plt.ylabel("Realized Frequency")
plt.title("Calibration Curves Comparison")
plt.legend()
plt.grid(True)

plt.show()

In [96]:
import os

print(os.ls())

In [ ]:
# Compute calibration curves
rv_true, rv_pred = calibration_curve(real_vol_results['y_test'],
                                     real_vol_results['test_probs'], n_bins=10)

iv_true, iv_pred = calibration_curve(imp_vol_results['y_test'],
                                     imp_vol_results['test_probs'], n_bins=10)

combo_true, combo_pred = calibration_curve(combo_results['y_test'],
                                           combo_results['test_probs'], n_bins=10)

all_ft_true, all_ft_pred = calibration_curve(all_ft_results['y_test'],
                                             all_ft_results['test_probs'], n_bins=10)

final_ft_true, final_ft_pred = calibration_curve(final_ft_results['y_test'],
                                                 final_ft_results['test_probs'], n_bins=10)

# Create 1x2 subplot layout
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

# ---- Left subplot: baseline models ----
ax = axes[0]
ax.plot(rv_pred, rv_true, marker='o', label='Real Vol Model')
ax.plot(iv_pred, iv_true, marker='o', label='Implied Vol Model')
ax.plot(combo_pred, combo_true, marker='o', label='Combined Model')
ax.plot(all_ft_pred, all_ft_true, marker='o', label='All Features Model')
ax.plot([0,1], [0,1], '--', color='gray', label='Perfect Calibration')

ax.set_title("Feature Sets Calibration Comparison")
ax.set_xlabel("Predicted Probability")
ax.set_ylabel("Realized Frequency")
ax.legend()
ax.grid(True)

# ---- Right subplot: final feature model ----
ax = axes[1]
ax.plot(final_ft_pred, final_ft_true, marker='o', color='purple', label='Final Features Model')
ax.plot([0,1], [0,1], '--', color='gray', label='Perfect Calibration')

ax.set_title("Final Feature Set Calibration")
ax.set_xlabel("Predicted Probability")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()